## Distributed Computing — PySpark Scaling Fundamentals

Moving a data pipeline from single-node (pandas/SQL) to a distributed engine (PySpark) isn't a syntax port — it's rethinking transformation order around data-movement cost, a cost pandas never had to consider on a single machine.

**Not a 1:1 Port — Lazy Evaluation and Shuffle-Aware Ordering**

1. Joins, aggregations, and window functions get rewritten against Spark's lazy-evaluation DataFrame API rather than translated method-for-method from pandas.
2. Transformation order gets restructured specifically to minimize shuffles — expensive cross-node data movement — which pandas never has to reason about on one machine.

In [ ]:
# Illustrative -- not executed (pyspark not installed here)
# pandas version: joins/filters execute immediately, in-memory, single machine
# df_pandas.merge(other_df, on="user_id").groupby("day").agg({"amount": "sum"})

# PySpark equivalent: lazy, builds a query plan first, executes distributed on .collect()/.write()
# df_spark.join(other_df, on="user_id").groupBy("day").agg(F.sum("amount"))
# -- filtering BEFORE the join (predicate pushdown) shrinks what gets shuffled across the join,
#    which is the kind of reordering that has no pandas equivalent to even think about

**What a Shuffle Actually Is, and Why It Dominates Cost**

1. A shuffle happens when data needs to move across nodes to bring matching keys together — for a `groupBy` or `join` on a key that isn't already co-located.
2. It's expensive because it means network I/O and disk spill, not just CPU work — minimizing shuffles (not just minimizing operation count) is the real optimization target in distributed pipelines.
3. Filtering to the relevant rows *before* a join (predicate pushdown), instead of after, is what actually produces most of the real-world overhead reduction from a migration like this.

In [ ]:
# Genuinely computable: rough shuffle-volume estimate for a groupBy/join,
# to reason about WHY reordering operations to filter-before-join reduces cost

def estimate_shuffle_bytes(n_rows, avg_row_bytes, filter_selectivity=1.0):
    """Rough estimate of data volume moved across the network during a shuffle."""
    rows_after_filter = int(n_rows * filter_selectivity)
    return rows_after_filter * avg_row_bytes

n_rows = 50_000_000
avg_row_bytes = 120

no_pushdown = estimate_shuffle_bytes(n_rows, avg_row_bytes, filter_selectivity=1.0)          # join first, filter after
with_pushdown = estimate_shuffle_bytes(n_rows, avg_row_bytes, filter_selectivity=0.08)        # filter first (8% selectivity), then join

print(f"Shuffle volume without predicate pushdown: {no_pushdown / 1e9:.2f} GB")
print(f"Shuffle volume with predicate pushdown:    {with_pushdown / 1e9:.2f} GB")
print(f"Reduction: {(1 - with_pushdown / no_pushdown):.1%}")

**Why Distribute at All, Instead of Scaling One Machine Up**

1. Vertical scaling has a hard ceiling — there's a largest machine you can rent or own, and single-machine pandas is also single-threaded for most operations regardless of available RAM.
2. Horizontal scaling (a cluster) has no such ceiling and is more cost-elastic, since a batch job's cluster only needs to scale up for the duration of the run.
3. Honest caveat: horizontal scaling isn't free — it introduces the shuffle-cost problem above that a single machine never had. The real tradeoff isn't "horizontal is strictly better," it's that data volume crosses a point where vertical scaling's ceiling and cost curve lose to horizontal's coordination overhead.

**Partition Count and Cluster Sizing**

1. Target a partition size in the 100–200MB range — small enough for parallelism across executors, large enough that per-partition overhead doesn't dominate.
2. Too few partitions underuses the cluster; too many adds scheduling overhead per task.
3. This is a starting heuristic, not a fixed rule — the right answer also depends on cluster core count (partitions should be a multiple of total cores for even scheduling) and is usually tuned empirically from actual job execution time.

In [ ]:
def recommend_partition_count(total_data_gb, target_partition_mb=150):
    total_mb = total_data_gb * 1024
    return max(1, round(total_mb / target_partition_mb))

for size_gb in [5, 50, 500]:
    n_partitions = recommend_partition_count(size_gb)
    print(f"{size_gb} GB dataset -> ~{n_partitions} partitions "
          f"(~{size_gb*1024/n_partitions:.0f} MB each)")

**Measuring the Improvement Honestly**

1. Measure wall-clock time for the specific stage that changed (e.g. data-prep), pre- vs. post-migration, on comparable data volume.
2. Isolate that stage from downstream steps so the number reflects the pipeline change itself, not unrelated infrastructure improvements happening at the same time — a claimed improvement that coincides with an unrelated cluster upgrade isn't attributable to the migration alone.